In [ ]:
import openai
from arguments import get_config
from interfaces import setup_LMP
from visualizers import ValueMapVisualizer
from envs.rlbench_env import VoxPoserRLBench
from utils import set_lmp_objects
import numpy as np
from rlbench import tasks

openai.api_base = "http://localhost:8317/v1"

openai.api_key = os.environ.get("OPENAI_API_KEY")  # set your API key in the environment

## Setup

In [ ]:
config = get_config('rlbench')
# uncomment this if you'd like to change the language model (e.g., for faster speed or lower cost)
# for lmp_name, cfg in config['lmp_config']['lmps'].items():
#     cfg['model'] = 'gpt-3.5-turbo'

# initialize env and voxposer ui
visualizer = ValueMapVisualizer(config['visualizer'])
env = VoxPoserRLBench(visualizer=None, camera_resolution=(256, 256))
lmps, lmp_env = setup_LMP(env, config, debug=False)
voxposer_ui = lmps['plan_ui']

In [ ]:
# env.load_task(tasks.PutRubbishInBin)

# descriptions, obs = env.reset()
# set_lmp_objects(lmps, env.get_object_names())


In [ ]:
# Perception backend is enabled in the instruction cell below so
# vlm_instruction always matches the current command.


In [ ]:
# obj = lmp_env.detect("rubbish")
# print("rubbish:", obj._position_world)

# obj = lmp_env.detect("bin")
# print("bin:", obj._position_world)

# obj = lmp_env.detect("tomato1")
# print("tomato1:", obj._position_world)

# obj = lmp_env.detect("tomato2")
# print("tomato2:", obj._position_world)


## Playground

By default we use one of the instructions that come with each task. However, you may treat each task as simply a scene initialization from RLBench, and feel free to try any task you can come up with for the scene.

Note:
- Whether an instruction can be executed or not depends on 1) whether relevant objects are available, and 2) capabilities of the overall algorithm.
- Each execution may produce one or more visualizations. You may view them in "./visualizations/" folder.
- The prompts are adapted with minimal change from the real-world environment in the VoxPoser paper. If a task failure is due to incorrect code generated by the LLM, feel free to modify the relevant prompt in "./prompts/" folder.
- You may view the reward by printing "env.latest_reward". These are computed by RLBench for each task.
- To inspect in viewer without performing any action, you may call "env.rlbench_env._scene.step()" in a loop.

In [ ]:
# # uncomment this to show all available tasks in rlbench
# # NOTE: in order to run a new task, you need to add the list of objects (and their corresponding env names) to the "task_object_names.json" file. See README for more details.
# print([task for task in dir(tasks) if task[0].isupper() and not '_' in task])

In [ ]:
# below are the tasks that have object names added to the "task_object_names.json" file
# uncomment one to use
env.load_task(tasks.PutRubbishInBin)
# env.load_task(tasks.LampOff)
# env.load_task(tasks.OpenWineBottle)
# env.load_task(tasks.PushButton)
# env.load_task(tasks.TakeOffWeighingScales)
# env.load_task(tasks.MeatOffGrill)
# env.load_task(tasks.SlideBlockToTarget)
# env.load_task(tasks.TakeLidOffSaucepan)
# env.load_task(tasks.TakeUmbrellaOutOfUmbrellaStand)

descriptions, obs = env.reset()
set_lmp_objects(lmps, env.get_object_names())  # set the object names to be used by voxposer

In [ ]:
# from perception.dump_rlbench_obs import dump_current_obs

# manifest = dump_current_obs(
#     env,
#     out_dir="/home/zhoudama/Voxposer-api/VoxPoser/src/perception_dumps/rubbish_01",
#     cameras=["front", "overhead", "wrist"],
# )
# manifest


In [ ]:
import os

# instruction = "throw away the rubbish"
instruction = np.random.choice(descriptions)
# os.environ["OPENAI_API_KEY"] = "..."  # set outside the notebook

env.enable_perception_backend(
    backend_type="inprocess",
    fallback_to_oracle=False,
    use_openai_vlm=True,
    vlm_model="gpt-5.4",
    vlm_base_url=openai.api_base,
    vlm_api_mode="responses",
    vlm_instruction=instruction,
)

voxposer_ui(instruction)


In [ ]:
# instruction = np.random.choice(descriptions)
# voxposer_ui(instruction)

In [ ]:
print(env.latest_reward)


In [ ]:
env.shutdown()
